# Assignment 2: Fine-tune a small language model (Qwen3-1.7B + LoRA)

**Narrow skill:** read a customer-support ticket (ru / kk / en / mixed) and answer with exactly one JSON object
`{"category": ..., "urgency": 0-2, "escalate": true/false}`. This is the Assignment 1 task, so the result can be compared with Jev.

**Runtime:** Runtime → Change runtime type → **T4 GPU**. **Key:** add `OPENROUTER_API_KEY` under Colab → Secrets (used to generate and check training data, and nothing else).

In [ ]:
%pip -q install unsloth openai
import os, sys
from google.colab import userdata
os.environ['OPENROUTER_API_KEY'] = userdata.get('OPENROUTER_API_KEY')
if not os.path.exists('slm.py'):
    # TODO: replace with your repo URL (or upload slm.py and data/ through the Colab file sidebar)
    !git clone https://github.com/YOUR_USER/YOUR_REPO.git repo && cp -r repo/* .
os.makedirs('results', exist_ok=True)
!nvidia-smi -L

## 1. Model choice

*(Draft, rewrite in your own words.)* I chose **Qwen3-1.7B**. It is on the allowed list, small enough to LoRA-train on a free T4 in minutes and to run in LM Studio on a laptop,
and Qwen3 is advertised as multilingual (119 languages), which matters for Russian and Kazakh tickets. The 4B version would likely be more accurate, but is slower to train, slower to run, and the task is narrow.

## 2. Task and test set

Test set: the 80 tickets from Assignment 1 with **your reviewed labels** (`data/test_tickets.csv`). They are never used for training.
Metrics: % valid JSON, route accuracy (escalate → `human`, otherwise the category), category / urgency / escalate accuracy, and exact match of all three fields.

In [ ]:
import unsloth                                   # must be imported before trl / transformers
from unsloth import FastLanguageModel
import gc, json, torch, pandas as pd
from datasets import Dataset
import slm as S
pd.set_option('display.max_colwidth', 140, 'display.width', 220)

test = S.load_test('data/test_tickets.csv')
print(len(test), 'test tickets;', test['lang'].value_counts().to_dict())

## 3. Dataset: generate with a bigger LLM, then clean

1. **Generate** with Claude Sonnet 5.5: one call per (category × urgency/escalate combination × language) cell, 10 tickets each, with the intended labels.
2. **Clean:** drop too short / too long texts, exact duplicates, near-duplicates (5-gram Jaccard ≥ 0.8) and anything too similar to a test ticket (≥ 0.6, to prevent leakage).
3. **Verify labels:** Claude labels every ticket blindly; a ticket is kept only if its label matches all three intended fields. This removes ambiguous or mislabeled examples.

In [ ]:
GEN_MODEL = 'anthropic/claude-sonnet-5.5'   # OpenRouter model id
client = S.make_client()
raw = S.generate_raw(client, GEN_MODEL, n_per_cell=10)       # 4 categories x 5 combos x 4 languages x 10 = 800 target
raw.to_csv('data/raw_generated.csv', index=False)
print(len(raw), 'raw tickets')

In [ ]:
train_df, stats = S.clean_dataset(raw, test, client=client, verify_model=GEN_MODEL, max_rows=1000)
print(json.dumps(stats, indent=2, ensure_ascii=False))
assert len(train_df) >= 300, 'fewer than 300 clean rows: raise n_per_cell above and rerun'
S.write_jsonl(train_df, 'data/train.jsonl')
S.write_jsonl(test.assign(instruction=test['text'])[['id','instruction','output','lang','category','urgency','escalate']], 'data/test.jsonl')
json.dump(stats, open('results/dataset_cleaning_stats.json', 'w'), indent=2, ensure_ascii=False)
sample = train_df.sample(20, random_state=1)
sample.to_csv('results/dataset_sample_20.csv', index=False)
sample

## 4. "Before": the base model on the test set

In [ ]:
MODEL_NAME = 'unsloth/Qwen3-1.7B-unsloth-bnb-4bit'
MAX_LEN = 1024

def load_base():
    return FastLanguageModel.from_pretrained(MODEL_NAME, max_seq_length=MAX_LEN, load_in_4bit=True)

def free_gpu():
    gc.collect(); torch.cuda.empty_cache()

model, tokenizer = load_base()
assert tokenizer.eos_token == '<|im_end|>'
FastLanguageModel.for_inference(model)
base_out, base_lat = S.run_model(model, tokenizer, list(test['text']))
base_metrics = S.evaluate(test, base_out, base_lat)
json.dump(base_out, open('results/before_outputs.json', 'w'), ensure_ascii=False, indent=1)
del model, tokenizer; free_gpu()
base_metrics

## 5. Fine-tune with LoRA (QLoRA: 4-bit base + LoRA adapters)

Loss is computed on the answer only (`train_on_responses_only`). The training text is exactly the inference prompt followed by the JSON answer, so train and test prompts match.

In [ ]:
from trl import SFTTrainer, SFTConfig
from unsloth import is_bf16_supported
from unsloth.chat_templates import train_on_responses_only

def train_and_eval(name, rows, r=16, alpha=16, lr=2e-4, epochs=2, seed=3407, keep_model=False):
    model, tokenizer = load_base()
    model = FastLanguageModel.get_peft_model(
        model, r=r, lora_alpha=alpha, lora_dropout=0, bias='none',
        target_modules=['q_proj','k_proj','v_proj','o_proj','gate_proj','up_proj','down_proj'],
        use_gradient_checkpointing='unsloth', random_state=seed)
    ds = Dataset.from_list([{'text': S.training_text(tokenizer, x['instruction'], x['output'])} for x in rows])
    hp = dict(name=name, model=MODEL_NAME, n_train=len(rows), lora_r=r, lora_alpha=alpha, learning_rate=lr, epochs=epochs,
              batch_size=8, grad_accum=2, optimizer='adamw_8bit', max_seq_length=MAX_LEN, seed=seed)
    args = SFTConfig(dataset_text_field='text', per_device_train_batch_size=8, gradient_accumulation_steps=2,
                     num_train_epochs=epochs, learning_rate=lr, warmup_steps=5, logging_steps=1, optim='adamw_8bit',
                     weight_decay=0.01, lr_scheduler_type='linear', seed=seed, report_to='none',
                     fp16=not is_bf16_supported(), bf16=is_bf16_supported(), output_dir=f'outputs_{name}')
    trainer = SFTTrainer(model=model, tokenizer=tokenizer, train_dataset=ds, args=args)
    trainer = train_on_responses_only(trainer, instruction_part='<|im_start|>user\n',
                                      response_part='<|im_start|>assistant\n<think>\n\n</think>\n\n')
    trainer.train()
    log = trainer.state.log_history
    if keep_model:
        model.save_pretrained(f'lora_{name}'); tokenizer.save_pretrained(f'lora_{name}')
    FastLanguageModel.for_inference(model)
    out, lat = S.run_model(model, tokenizer, list(test['text']))
    metrics = S.evaluate(test, out, lat)
    del model, tokenizer, trainer; free_gpu()
    return dict(hp=hp, log=log, outputs=out, metrics=metrics)

rows = S.read_jsonl('data/train.jsonl')
main = train_and_eval('main', rows, r=16, alpha=16, lr=2e-4, epochs=2, keep_model=True)
json.dump(main['hp'], open('results/hyperparameters_main.json', 'w'), indent=2)
json.dump(main['outputs'], open('results/after_outputs.json', 'w'), ensure_ascii=False, indent=1)
S.plot_loss(main['log'], 'results/loss_curve_main.png', 'Training loss (main run)')
from IPython.display import Image; Image('results/loss_curve_main.png')

## 6. Evaluation: before vs after

In [ ]:
table = pd.DataFrame({'base (before)': base_metrics, 'fine-tuned (after)': main['metrics']}).T
table.to_csv('results/before_after_metrics.csv')
display(table.round(3))
cmp10 = S.side_by_side(test, base_out, main['outputs'])
cmp10.to_csv('results/side_by_side_10.csv', index=False)
cmp10

## 7. One experiment: half the training data

Everything is identical to the main run except the number of training examples (50%, random, fixed seed). Question: does more data help?

In [ ]:
import random
random.Random(0).shuffle(rows)
exp = train_and_eval('half_data', rows[:len(rows)//2], r=16, alpha=16, lr=2e-4, epochs=2)
json.dump(exp['hp'], open('results/hyperparameters_experiment.json', 'w'), indent=2)
S.plot_loss(exp['log'], 'results/loss_curve_experiment.png', 'Training loss (half data)')
exp_table = pd.DataFrame({'main (100% data)': main['metrics'], 'experiment (50% data)': exp['metrics']}).T
exp_table.to_csv('results/experiment_metrics.csv')
exp_table.round(3)

## 8. Export to GGUF

Download the `.gguf` file, then follow the LM Studio steps in `README.md` and take the screenshot.

In [ ]:
import glob
model, tokenizer = FastLanguageModel.from_pretrained('lora_main', max_seq_length=MAX_LEN, load_in_4bit=True)
model.save_pretrained_gguf('gguf_model', tokenizer, quantization_method='q4_k_m')
files = glob.glob('gguf_model*/**/*.gguf', recursive=True) + glob.glob('*.gguf')
print(files)
from google.colab import files as colab_files
colab_files.download(sorted(files, key=os.path.getsize)[0])   # smallest .gguf = the q4_k_m file

## 9. Bonus (+10): fine-tuned model vs Jev

Put the Assignment 1 `results_table.csv` at `data/a1_results_table.csv`. Latency below is measured here on a Colab T4 (batch size 1); cost is the
GPU time per decision at an assumed hourly GPU price. **Set `GPU_USD_PER_HOUR` to the real price you want to assume** (use 0 for your own machine).

In [ ]:
GPU_USD_PER_HOUR = 0.35     # ASSUMPTION: replace with a price you can cite; 0 = running on hardware you already own
a1 = pd.read_csv('data/a1_results_table.csv', index_col=0)
m = main['metrics']
slm_cost = m['latency_mean_s'] * GPU_USD_PER_HOUR / 3600 * 1000
bonus = pd.DataFrame({
    'route_acc': [a1.loc['Jev','route_acc'], a1.loc['anthropic/claude-sonnet-5.5','route_acc'], m['route_acc']],
    'latency_mean_s': [a1.loc['Jev','latency_mean_s'], a1.loc['anthropic/claude-sonnet-5.5','latency_mean_s'], m['latency_mean_s']],
    'cost_per_1k_usd': [a1.loc['Jev','cost_per_1k_usd'], a1.loc['anthropic/claude-sonnet-5.5','cost_per_1k_usd'], slm_cost],
}, index=['Jev', 'anthropic/claude-sonnet-5.5', 'Qwen3-1.7B fine-tuned'])
bonus.to_csv('results/bonus_comparison.csv'); bonus.round(4)

## 10. Conclusion (write this yourself)

*3-5 sentences: what improved, what did not, and why. Use the before/after table, the 10-prompt comparison, the loss curve and the experiment result.*